# 02 — MediaPipe Holistic feature extraction (Kaggle CPU)

**Purpose.** Extract the same features as the pilot (VIDEO mode, 15 fps grid, `feat_layout_v1`, no face mesh or image stored) for all 400 clips, one folder per split, with QC, failures and hand coverage. Resume-safe.

**Kaggle settings.** Accelerator **None** (MediaPipe runs on the CPU here; this saves GPU quota) · Internet **On** (pip + model).

**Inputs.** Dataset `bdsl-k400`. **If a previous version of this notebook stopped early:** also *Add Input* → that version's Output; finished clips are then skipped.

**Outputs** (under `/kaggle/working/k400/`, saved by *Save Version → Save & Run All*). `processed/features/{train,val,test}/<Index>.npz`, `processed/qc/qc.csv`, `failures.csv`, `hand_coverage_summary.csv`, `extract_run.json`.

**Expected run time** [ESTIMATE, not measured yet]. 30–90 minutes for 400 clips (≈ 42,000 frames; desktop pilot rate 34.6 frames/s with 5 processes; Kaggle CPU speed unknown — `extract_run.json` records the real time).

**What to check.** `EXTRACTION COMPLETE`, 400/400 clips, failures 0, privacy OK True, rotation OK True. If it says NOT COMPLETE because of the time budget: Save Version, then run a new version with this version's Output added as Input.

> Honesty rules: label every number measured or estimated; results are small-scale and signer-dependent;
> the test split is used once (notebook 07); low coordinate error is not proof of correct sign language;
> G0 glosses are pseudo-glosses, not BdSL annotation; no human or expert validation exists.

In [ ]:
# Cell 1 - Setup (the same in every notebook). Finds the data and the code, sets the paths, and copies the
# outputs of earlier notebooks (added with "Add Input") into /kaggle/working/k400 so results accumulate.
import os, sys, zipfile, pathlib, json
NOTEBOOK = "02_extract_features"

def _find_root():
    base = pathlib.Path("/kaggle/input")
    if base.is_dir():
        for dirpath, dirnames, filenames in os.walk(base):
            if "UPLOAD_MANIFEST.csv" in filenames and "PACKAGE_INFO.json" in filenames:
                return pathlib.Path(dirpath)
            depth = len(pathlib.Path(dirpath).relative_to(base).parts)
            dirnames[:] = [] if depth >= 4 else [d for d in dirnames if d not in ("videos", "k400")]
        zips = sorted(base.rglob("bdsl-k400-*.zip"))
        if zips:  # Kaggle kept the upload as one zip file: unpack it once (a few minutes)
            dest = pathlib.Path("/tmp/k400_dataset")   # /tmp is not saved as Output
            if not (dest / "UPLOAD_MANIFEST.csv").is_file():
                print("Unpacking", zips[0], "->", dest)
                zipfile.ZipFile(zips[0]).extractall(dest)
            return dest
        raise FileNotFoundError("The private dataset is not attached. Right panel: Add Input -> Your Datasets -> bdsl-k400.")
    here = pathlib.Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "pyproject.toml").is_file() and (c / "kaggle" / "src" / "k400").is_dir():
            return c
    raise FileNotFoundError("Open this notebook inside the project folder, or on Kaggle with the dataset attached.")

DATASET_ROOT = _find_root()
CODE_ROOT = DATASET_ROOT / "code" if (DATASET_ROOT / "code" / "pyproject.toml").is_file() else DATASET_ROOT
for _p in (CODE_ROOT / "kaggle" / "src", CODE_ROOT / "src"):
    sys.path.insert(0, str(_p))
from k400.setup import setup
S = setup(DATASET_ROOT, CODE_ROOT, NOTEBOOK)

## Settings

In [ ]:
MAX_MINUTES = 300   # no new clip is started after this; the version then finishes and saves
WORKERS = None      # None = one process per CPU core

## Step 1 — MediaPipe 0.10.35 (main route, else the pinned fallback)

In [ ]:
rc_mp = S.k400("mp-setup")
mp = json.load(open(S.state_root / "outputs/env/mediapipe_route.json"))
assert mp["ok"], "MediaPipe failed on both routes: stop and paste the output to Claude"
EXTRACT_PY = mp["python"]
print("extraction runs with:", EXTRACT_PY)

## Step 2 — Extraction (prints progress; safe to resume)

In [ ]:
args = ["extract", "--max-minutes", str(MAX_MINUTES)] + (["--workers", str(WORKERS)] if WORKERS else [])
rc_extract = S.k400(*args, python=EXTRACT_PY)

## Step 3 — Quality control

In [ ]:
import pandas as pd
from IPython.display import display
run = json.load(open(S.state_root / "processed/qc/extract_run.json"))
print(json.dumps({k: run[k] for k in ("feat_ver", "mediapipe", "platform", "clips_with_features", "clips_expected", "failures_or_missing", "privacy_ok_all", "rotation_ok_all", "seconds_this_run", "stopped_by_time_budget")}, indent=1))
display(pd.read_csv(S.state_root / "processed/qc/hand_coverage_summary.csv"))
fails = pd.read_csv(S.state_root / "processed/qc/failures.csv")
print("failures / missing:", len(fails))
display(fails.head(20))

In [ ]:
S.finish(checks=[
    ("all 400 clips have features", run["clips_with_features"] == 400),
    ("no failures", run["failures_or_missing"] == 0),
    ("privacy check: no face mesh / image stored", run["privacy_ok_all"]),
    ("rotation: decoded size = manifest display size", run["rotation_ok_all"]),
    ("feat_ver equals the pilot (same MediaPipe, model, settings)", run["feat_ver"] == "fa89892d4908"),
])